# Adapting HRNet for Cephalometric Landmark Detection
This notebook documents the specific architectural and mathematical decisions made to transition the model from producing poor, dislocated blocky predictions to highly accurate facial landmark representations. 

We address the failures in feature representation, geometry destruction, gradient wash-out, and integer discretization by aligning closely with the official HRNet methodologies.

## 1. The Architecture Fix: Preserving Stride 4 High-Resolution Streams
**The Problem**
Initially, using `timm.create_model('hrnet_w18', features_only=True)` extracted a single layer from the backbone which defaulted to a stride 2 (or heavily pooled classification stride). This destroyed the parallel multi-resolution branches that give "High-Resolution Net" its name. Landmarks like the Pronasal point are extremely delicate; downsampling the image entirely destroyed those spatial cues.

**The Decision**
We bypassed `features_only` and manually instantiated the backbone to access `.stages(x)`. By grabbing the output of the final stage, we get 4 parallel feature maps at channel dimensions `[18, 36, 72, 144]`. 

**Why It Worked**
Instead of predicting on a tiny pooled image representation, we manually upsampled all 4 layers back to the stride 4 reference (128x128 for a 512x512 image) using `bilinear` interpolation, yielding a deep 270-channel volumetric feature map. This retained both the rich deep semantics (for identifying *what* part of the face it is) and the raw spatial dimensions (for identifying *exactly where* the pixel lives) for the $1\times1$ convolution prediction head to output the point.

## 2. Transformation Fix: Defending Medical Aspect Ratios
**The Problem**
A common bottleneck in image training is resizing rectangular original images to a standard model size (like 512x512) directly using `cv2.resize`. Doing this squashes horizontal images and stretches vertical ones. When the model stretches a skull radiograph, it changes the natural anatomical angles between the nose, lips, and chin.

**The Decision**
We utilized the core `crop_v2` and `transform_pixel` affine transformations directly from the HRNet research team. We convert the coordinate bounding boxes into a standardized `center` point and `scale` (calculating scale by padding `max(w, h) / 200.0 * 1.25`). 

**Why It Worked**
Because affine transformations maintain isometric scaling, the face is cropped uniformly before padding is applied. The resulting 512x512 square tensor is a mathematically perfect window protecting the patient's organic anatomical curvature layout, ensuring the neural network doesn't have to learn distorted edge variations.

## 3. Loss Fix: Preventing Gradient Wash-out (Weighted MSE masks)
**The Problem**
Our coordinate ground-truth maps are built by drawing tiny 5-pixel radius spheres of Gaussian heat onto an empty `128x128` grid. This results in $16,384$ pixels where roughly $99\%$ of them are just the number $0.0$. Standard `nn.MSELoss(output, target).sum()` calculates the errors across the entire grid uniformly. As those background 0 values are very easy to predict, the loss rapidly plunges leaving the actual gradient of the tiny visible landmark buried and unoptimized. 

**The Decision**
We built a custom `WeightedMSELoss` that pulls in a `target_weight` boolean mask map during Dataloading checks (dictating `0` if invisible or out-of-bounds, and `1` if visible).

**Why It Worked**
During computing: `criterion(prediction, ground_truth) * target_weight`, the gradients perfectly collapse the dead, empty space and strictly penalize the model exclusively for where it fails to trace the localized anatomical bumps, forcing localized high-frequency convergence.

## 4. Inference Fix: Sub-Pixel Expansion Mapping
**The Problem**
Early on in the project, the outputs were decoded back to numerical format via pure integer index finding, ex: `argmax(heatmaps)`. So if the landmark peak landed on pixel `[85, 30]`, it outputted those integers. The issue is these are low-resolution $128\times128$ heatmaps, whereas real input images are thousands of pixels wide. Mapping a rigid integer mapping back upward to massive sizes creates blocky artifacts—sometimes missing real features by 5+ integer steps.

**The Decision**
We threw out `argmax` strings entirely and adopted HRNet's `decode_preds` framework to trace the slope of the localized prediction output.

**Why It Worked**
`decode_preds` doesn't just read the single maximum integer point. It reads the maximum point AND the neighbors on a logarithmic slope around it, creating a localized Taylor Expansion. This tells us the *fractional* top of a curve (i.e. `x: 85.341`). This fractional coordinate is then reversely multiplied by the exact affine `scale` modifier saved during Dataset generation mapping the decimal coordinates symmetrically back onto the large, uncropped raw image coordinate mapping space!